<a href="https://colab.research.google.com/github/Living-with-machines/dhoxss-text2tech/blob/prompting/Sessions/5b-PromptingLLMs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prompting (Open-Source) LLMs for Humanities Research

Welcome! This notebook introduces **prompt engineering**: the principal way of interacting with Large Language Models (LLMs). 

We start with the basics of writing good prompts, then use those skills to turn an LLM into a **research assistant** that reads, cleans, classifies and structures historical newspaper articles.

This introduction is inspired by:
- **["Prompt Engineering for Generative AI"](https://www.oreilly.com/library/view/prompt-engineering-for/9781098153427/)** by James Phoenix and Mike Taylor
- **["Prompt Engineering Guide"](https://github.com/dair-ai/Prompt-Engineering-Guide?tab=readme-ov-file)**
- **[Einführung in das Prompt Engineering](https://agki-dh.github.io/pages/webinar/page-3.html)** by Christopher Pollin
- Paul Fyfe's *["By Accident or Design"](https://global.oup.com/academic/product/by-accident-or-design-9780198732334)*
- The Hugging Face [structured generation cookbook](https://huggingface.co/learn/cookbook/structured_generation)

**Q:** What does "prompt engineering" mean to you? 

**Q:** What would you like an AI research assistant to help you with?

## What we'll cover

**Part A — Prompt Engineering Basics**
1. Open-source vs. closed models, and the Hugging Face Hub
2. Loading a model that adapts to your hardware (CPU or GPU)
3. Prompt structure: system and user messages
4. Core principles: direction, format, examples, evaluation, dividing labour
5. Controlling generation: temperature and other parameters

**Part B — LLMs as Research Assistants**
6. A "baby RAG" pipeline for historical newspapers
7. Classifying and cleaning noisy OCR text
8. Structured generation: turning prose into data
9. (Optional/Advanced) GraphRAG: from text to knowledge graph
10. What to do when small open models aren't good enough


# Part A — Prompt Engineering Basics

## What is Prompt Engineering?

**Prompt engineering** amounts to crafting *inputs* (prompts) that **guide** or **instruct** AI models to generate desired *outputs*. The messages we send to a model are called "prompts".

Prompt engineering isn't really about complex coding — it's about clear communication, critical thinking, and understanding how a model processes information, so you can anticipate its limitations and (as much as possible) avoid its biases.

## Open vs. closed models

![](https://i.imgflip.com/76eq2p.jpg)

Many well-known AI tools are proprietary and **"closed"**. Closed models are usually very easy to use, but convenience often comes at a literal cost. They're great for **prototyping**, but not always ideal for working at scale with (potentially sensitive) research data.

The Open Source Initiative recently published an [Open Source AI Definition — OSAID 1.0](https://opensource.org/ai/open-source-ai-definition ).


![open](https://raw.githubusercontent.com/Living-with-machines/dhoxss-text2tech/refs/heads/main/Sessions/images/openness.png)
**Open-source** models can take more effort to set up, but offer real benefits for research:

- **Privacy**: you don't have to send your data (or your unpublished research ideas) to a company.
- **Cost**: running an open model can be much cheaper if you want to apply a prompt to, say, 10,000 newspaper articles.
- **Transparency**: openness comes in degrees — even when you can download a model's weights, you may still know little about its training data.
- **Customization**: open models can be fine-tuned on specialised data (historical texts, library catalogues, etc.).

Two popular ways to access open models are the **[Hugging Face](https://huggingface.co/) Hub** (a community-driven hub of models, datasets and libraries — our focus today) and **[Ollama](https://ollama.com/)** (built for running models easily on your own machine).


## Setting up: install libraries

In [ ]:
# Colab already ships with torch, pandas, matplotlib, networkx and ipywidgets.
# We just make sure transformers (and friends) are recent enough to know about
# the models we'll use below.
# !pip install -q -U transformers accelerate huggingface_hub

## Loading a model that fits your hardware

Depending on your subscription and use you can access hardware acceleration on Colab.

> Go to `Runtime` → `Change runtime type` → select a `T4 GPU`. A CPU runtime works fine too — the notebook picks a smaller model automatically in that case.

Rather than hard-coding one model, we detect whether a GPU (CUDA) is available and **pick the model size accordingly**:

- No GPU → `Qwen/Qwen2.5-0.5B-Instruct`: small enough to run comfortably on CPU. But you still have to be patient! It will take around one minute to generate a response, and it might be rubbish!
- GPU available → `Qwen/Qwen2.5-3B-Instruct`: bigger and more capable, using the extra speed and memory a GPU provides.

We use Alibaba's **Qwen2.5** instruct models: they are strong, open models at these sizes, and — unlike some other model families (Llama, Gemma) — **openly licensed with no sign-up, terms-of-use click-through, or access token required**. That means this cell should "just work" the moment you run it, on Colab or your own machine.

The same pattern — checking `torch.cuda.is_available()` and branching on it — is a useful one to reuse in your own projects: it lets a single notebook run unmodified on your laptop, a lab machine, or Colab.


In [ ]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")

if cuda_available:
    device = "cuda"
    model_id = "Qwen/Qwen2.5-3B-Instruct"
else:
    device = -1  # -1 tells the pipeline to use the CPU
    model_id = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Using model: {model_id} (device: {device})")

In [ ]:
print(f"Loading {model_id}. This can take a moment the first time...")

# padding_side='left': needed to correctly batch-generate with a decoder-only model
# (see apply_completions() in Part B, which sends several prompts to the model at once)
tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16 if cuda_available else torch.float32,
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device,
)

print("Model loaded ✅")

We'll also write one small helper function, `get_completion()`, that we will reuse for the **rest of this notebook**: it takes a list of chat messages (see "Prompt Structure" below) and returns the model's reply as plain text.

In [ ]:
def get_completion(messages: list, max_new_tokens: int = 256,
                    temperature: float = 0.1, top_p: float = 0.9) -> str:
    """Send a list of chat messages to the loaded LLM and return the reply as text.

    Arguments:
      messages (list): a list of {"role": ..., "content": ...} dictionaries,
        e.g. a "system" message followed by a "user" message.
      max_new_tokens (int): how much text the model is allowed to generate.
      temperature (float): regulates the "creativity" of the generation.
      top_p (float): cumulative probability considered during generation.
    """
    output = generator(
        messages,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
    )
    return output[0]["generated_text"][-1]["content"]

## First contact: a simple prompt

In [ ]:
# return_full_text=False: only show what the model added, not our prompt echoed back
print(generator('What is the colour of the sky?', return_full_text=False)[0]['generated_text'])

**✏️ Exercise**

Ask the model something else — for example, whether pineapple is allowed as a pizza topping!

In [ ]:
# Enter code here

## LLMs are open to manipulation: ``EmoPrompt''

Any change in wording — even something that shouldn't matter logically — can noticeably change what a model produces. This matters a great deal for research: if your results depend on incidental phrasing, that's worth knowing (and reporting).

A very simple example of prompt "engineering" is the addition of "emotional" cues to indicate how important the answer is to you!

In [ ]:
print(generator('Write a very short poem.', return_full_text=False)[0]['generated_text'])

In [ ]:
print(generator('Write a very short poem. I will tip you £1000 for an example that makes me cry!', return_full_text=False)[0]['generated_text'])

**✏️ Exercise**

Can you think of other ways to nudge the model (flattery, urgency, a fake deadline)? Try one. Then try something unrelated to persuasion, e.g. ask it to count the number of "r"s in *"I drive my scooter to the library"* — does it get it right?

In [ ]:
# Enter code here

## Prompt structure: system and user messages

When 'chatting' with an LLM, we usually send (at least) two messages:

**System message** — sets the scene for the whole conversation:
- **Behaviour**: how should the model act (helpful, neutral, formal) or what role should it play ("a reference librarian", "a literary critic")?
- **Constraints**: what should it avoid, or how should it format its answers?
- **Context**: background information that should stay constant.

**User message** — the specific request for this turn:
- **Query**: the question, instruction, or text the model needs to respond to.
- **Dynamic**: this is what changes from one call to the next.

The Hugging Face chat template represents this as a list of dictionaries:

```python
messages = [
    {"role": "system", "content": "<system prompt here>"},
    {"role": "user",   "content": "<user prompt here>"},
]
```

### A "universal" system prompt?

From [Jeremy Howard](https://x.com/jeremyphoward/status/1689464589191454720?lang=de):

*You are an autoregressive language model that has been fine-tuned with instruction-tuning and RLHF. You carefully provide accurate, factual, thoughtful, nuanced answers, and are brilliant at reasoning. If you think there might not be a correct answer, you say so.*

Let's give our poet a persona on top of that:

In [ ]:
messages = [
    {
        "role": "system",
        "content": """
          You are an autoregressive language model that has been fine-tuned with instruction-tuning and RLHF.
          You carefully provide accurate, factual, thoughtful, nuanced answers, and are brilliant at reasoning.
          If you think there might not be a correct answer, you say so. If you don't know the answer, you say so.

          You are also a brilliant poet who composes the best poems the world has ever seen.
          Your poems resemble Shakespeare and Homer, but are written in cockney English.
          """,
    },
    {"role": "user", "content": "Write a poem about London on a rainy day."},
]

print(get_completion(messages, temperature=1.0))

**✏️ Exercise**

1. Give the model an Italian persona. What does it think of pineapple as a pizza topping?
2. Rewrite the poet's system prompt: try a different language, poetic style, or persona.

In [ ]:
messages = [
    {"role": "system", "content": """<write system prompt here>"""},
    {"role": "user", "content": "Write a poem about London on a rainy day."},
]

print(get_completion(messages, temperature=0.7))

## Core principles of effective prompt engineering

Drawing on best practices, here are foundational ideas for crafting better prompts.

### 1. Give direction

- **Be task-specific**: instead of "Tell me about Shakespeare", try "Summarise the main themes in *Hamlet* for a college-level literature class, focusing on revenge and madness." Use concrete verbs — *summarise*, *extract*, *explain*.
- **Define a persona**: "You are a reference librarian creating metadata for a collection of medieval manuscripts."
- **Provide context**: what does the model need to know, and what will the output be used for?

**✏️ Exercise: Named Entity Recognition**

Below is a snippet from a 19th-century newspaper (with typical OCR noise), and a deliberately *under-specified* prompt. Improve it by giving the model more direction:
- What task should it perform?
- What persona suits the task?
- Should these instructions go in the system or the user message?
- Try it on a different snippet too (see the RAG section below for more real examples).

In [ ]:
text = """POOR T,i,ENIPAT A 1„k CT  The Poor Law Coirdnissioti(rs have issued a ei; cular,
dated the 20th instant, stating that they have consulted the Attorney and
Solicitor-General on the construction of the late Removal Act, and give as the
result:— I. " That the proviso to the Ist section of the 9 and 10 Vict., c. 66,
which sets forth the exceptions to the principal enactments that are to be
excluded in the computation of time, is net retrospective in its operation, so
as to apply to cases where the five years' residence was complete before the statute.
2. " That an interval between the completion of the five years residence and the
application for the warrant of removal filled up by one of the exceptions contained
in the proviso will not p event the operation of the statute in restraining the
removal of the pauper whu had resided for the specified time. 3. " That orders
of removal obtained previous to th• passing of the Act, but not then executed
by the removal of the paupers."""

In [ ]:
messages = [
    {"role": "system", "content": "<enter system prompt>"},
    {"role": "user", "content": f"Who is mentioned below? <change user prompt>\n\n{text}"},
]

print(get_completion(messages, max_new_tokens=250))

### 2. Specify format and structure

- Specify the format of the **input**: "look only at the text between `###`".
- Specify the structure of the **output**: a bulleted list? YAML? Plain text? Say so explicitly.

Asking the model to follow a fixed structure makes it much easier to process the response programmatically later. Below, we ask for **[YAML](https://yaml.org/)**, a human-readable format for structured data:

```yaml
name: Alice
age: 30
active: true
```

In [ ]:
messages = [
    {
        "role": "system",
        "content": """You are an information extraction system.
         Extract all named entities from the text demarcated by triple hashtags, i.e. ###.
         Return them in valid YAML format.""",
    },
    {
        "role": "user",
        "content": f"""Extract all named entities from the following text and return them in valid YAML.
          Group entities by type using these categories (only include ones that appear in the text):
          PERSON, ORGANIZATION, LOCATION, DATE, EVENT, LAW, OTHER

          ###{text}###""",
    },
]

output = get_completion(messages, max_new_tokens=250)
print(output)

### 3. Give examples

Providing examples usually helps the model produce more accurate, consistently formatted output. Each example is called a "shot" in LLM terminology.

**Zero-shot** learning is when the model performs a task from **instructions alone**, with no examples. Suppose we ask it to create [Dublin Core](https://www.dublincore.org/) metadata — a standard librarians and archivists use — without showing it what one looks like:

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful metadata specialist and librarian."},
    {
        "role": "user",
        "content": """Create Dublin Core metadata elements for the following item:
        A digitized oral history interview with a retired steelworker recorded in Pittsburgh in 1982.""",
    },
]

print(get_completion(messages, max_new_tokens=250))

Zero-shot learning is easy to set up, but asks the model to do a lot with very little to go on — it has to rely purely on what it already "knows" about Dublin Core.

**Few-shot** learning gives the model a small number of worked examples that demonstrate the expected structure, vocabulary and level of detail:

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful metadata specialist and librarian who converts records to Dublin Core."},
    {
        "role": "user",
        "content": """
Example 1:
Record: Johnson, Alice, 'Memories of the Textile Mills', Textile industry — Oral histories (1975)
Dublin Core:
dc:title: Memories of the Textile Mills
dc:creator: Johnson, Alice
dc:subject: Textile industry — Oral histories
dc:date: 1975

Example 2:
Record: Building the Golden Gate Bridge, California Department of Transportation (1937)
Dublin Core:
dc:title: Building the Golden Gate Bridge
dc:creator: California Department of Transportation
dc:subject: Bridges — Construction — California
dc:date: 1937

Now create Dublin Core metadata elements for the following item:
A digitized oral history interview with a retired steelworker recorded in Pittsburgh in 1982.""",
    },
]

print(get_completion(messages, max_new_tokens=250))

**✏️ Exercise**

1. Build an **emotion classifier**: write a new system prompt and give the model a few examples of how to assign an emotion label to a fragment of text.
2. Bonus: think of another small classification task you could set up with few-shot examples.

In [ ]:
messages = [
    {"role": "system", "content": "<enter system prompt here>"},
    {
        "role": "user",
        "content": """<enter user prompt here>
        <specify the task>
        <example 1>
        ...
        <example n>

        <example to classify>""",
    },
]

print(get_completion(messages, max_new_tokens=250))

### 4. Divide labour

A powerful strategy is breaking complex work into smaller, more manageable steps — the same way you'd tackle a complex research problem more generally.

**Chain-of-thought prompting** encourages the model to work through a problem step by step instead of jumping to conclusions. Instead of "What are the main themes in this text?", try "What are the main themes in this text? Let's think step by step" — or spell out the steps yourself. This is especially useful for tasks that require nuanced reasoning, and it lets you spot (and correct) flawed assumptions in the model's thinking.

In [ ]:
text = """
FRIGHTFUL ACCIDENT UN THE NORTH EASTERN RAILWAY. On Monday last, a frightful accident occurred on this line of railway. The first class express train left Ed in bro', as usual at 9-50 a.m. It consisted of au engine, next to which was a cuard's van, four car- nages, -and another break van which was attached to the end of the train. All went well until the train arrived within about eight miles af Newcastle, When the tire of one of the wheels of the last passenger carriage gave way, and threw it off the sine. Tbe guard immediately applied his break, but the engine-driver took no notice whatever, and the train proceeded at the rate of nearly sixty miles an hour. The guard's van, on the break -being screwed tight, broke tbe coupling chains) aud was left on the line. The carriage, however, ' Which Was off the way, and which contained Lady Ferguson Davis, and her son, Mr. Aid. Chadwick, of York; the Rev. Mr. All- good, Mr. Alfred E. Hargrove, oue of the proprietors of the York Herald, and Mr. J. Agar, ot York, was dragged along for three miles, at full speed, until very near Rillingworth station, when the axles were knocked away, the coupling chains broken, and tbe carriage released from the train. The passengers fortunately escaped with a few bruise*, but the crrriage was a complete wreck, the bottom being knocked out an I the fittings destroyed. The line was strewed for three miles with portious of the broken carriage and passengers' luggage, which had fallen through the bottom of the carriage during its progress over the stones and sleepers. Labourers were at once set to work to clear away the broken carriage, and the passengers, whose alarm may be better imagined than described, were forwarded to Newcastle. This accident affords convincing proof of the urgent necessity that exists for some means being adopted so as to enable guards to communicate with tbe engine driver ; for, if any plan of the kind had been in operation on this train, the accident might have been avoided altogether.
"""

messages = [
    {"role": "system", "content": "You are a helpful analyst of historical media. Read the text with attention to detail. Do not make stuff up!"},
    {"role": "user", "content": f"Explain the accident described in the text between ###. Think step-by-step.\n\n###{text}###"},
]

print(get_completion(messages, max_new_tokens=500))

A related idea is **prompt chaining**: breaking a large task into a *sequence* of smaller prompts, where each one builds on the previous output — an analytical pipeline where each stage does one job. For example, when analysing a historical document:

1. "Summarise the key factual claims in this document."
2. "Given these claims [paste output], which would have been controversial in their historical context?"
3. "Based on these controversial claims [paste output], what does this reveal about the document's intended audience?"

This gives you more control over each step, lets you course-correct mid-process, and lets you reuse a successful chain across similar material. We'll build something similar — a small pipeline over hundreds of newspaper snippets — in Part B.

**✏️ Exercise**

Find a short poem (e.g. from the [Gutenberg Project](http://gutenberg.org/ebooks/bookshelf/637)). Ask the model to act as a literary critic and analyse it. Then try adding "think step-by-step" — does the analysis improve?

In [ ]:
# Enter code here

### 5. Evaluate quality

A critical (and often skipped!) step is systematically checking whether the LLM is actually doing what you need. Unlike traditional software, LLM output is not fully predictable, so ongoing assessment matters. Which approach fits depends on your project's scale and stakes:

- **Benchmarks**: standardised tests against a "ground truth" (see the Hugging Face [Open LLM Leaderboard](https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard#/)). Useful when your task matches an established benchmark and you need to justify your methodology — but most benchmarks target technical domains and rarely capture the interpretive nuance humanities work requires.
- **Vibe evaluation / eyeballing**: reading outputs and judging whether they "feel right". Good for exploratory work and small projects, where an expert can recognise good analysis on sight.
- **Annotation experiments**: define explicit quality criteria and systematically rate outputs against them (e.g. "Does it accurately reference the text? Yes/No", "Interpretive insight? 1–5"). Worth the extra effort when you're processing large volumes, need to demonstrate rigour, or are comparing prompting strategies (A/B testing).

Let's try a lightweight annotation exercise: generate a batch of short poems, then rate them ourselves.

In [ ]:
nouns_list = ['Sun', 'Tree', 'House', 'Car', 'Ocean', 'Mountain', 'Cloud', 'River', 'Book', 'Bird']

prompt = 'Write a short poem about a {noun}.\n'
# passing a list of prompts (rather than calling generator() ten times) lets the
# model process them together in batches, which is much faster
poems = generator([prompt.format(noun=n) for n in nouns_list], max_new_tokens=80,
                   return_full_text=False, batch_size=8)

In [ ]:
import pandas as pd

poems_df = pd.DataFrame([poem[0]['generated_text'] for poem in poems], columns=['response'])
poems_df['feedback'] = pd.Series(dtype='str')
poems_df.head()

In [ ]:
import ipywidgets as widgets
from IPython.display import display

response_index = 0

def on_button_clicked(b):
    global response_index
    # convert thumbs up / down to 1 / 0
    poems_df.at[response_index, 'feedback'] = 1 if b.description == "\U0001F44D" else 0
    response_index += 1
    if response_index < len(poems_df):
        update_response()
    else:
        poems_df.to_csv("poem_ratings.csv", index=False)
        response.value = "<p><em>All done — ratings saved to poem_ratings.csv</em></p>"

def update_response():
    new_response = poems_df.iloc[response_index]['response']
    response.value = f"<p>{new_response}</p>" if pd.notna(new_response) else "<p>No response</p>"
    count_label.value = f"Response: {response_index + 1}/{len(poems_df)}"

response = widgets.HTML()
count_label = widgets.Label()
update_response()

thumbs_up = widgets.Button(description='\U0001F44D')
thumbs_up.on_click(on_button_clicked)
thumbs_down = widgets.Button(description='\U0001F44E')
thumbs_down.on_click(on_button_clicked)

display(response, widgets.HBox([thumbs_down, thumbs_up]), count_label)

**✏️ Exercise**

Change the poem prompt (a different style, a persona, a constraint on length) and re-run the two cells above. Do your annotations improve?

## Controlling generation: parameters

Prompting steers the model towards the response you want — but you can also control **how** it generates text. The most useful parameters for humanists to know:

1. **`temperature`**: controls "randomness"/"creativity".
   - **Low** (0.1–0.3): the model favours the most likely next words — good for factual tasks, extraction, or consistent formatting.
   - **High** (0.7–1.5): the model takes more risks — good for poetry or brainstorming, but more prone to hallucination or incoherence.
2. **`do_sample`**: must be `True` to use `temperature` at all. If `False`, generation is deterministic — the same prompt always yields the same output.
3. **`max_new_tokens`**: simply caps how much text is generated.
4. **`top_k`/`top_p`** *(advanced)*: prune the vocabulary the model samples from. `top_k` keeps only the *k* most likely next words; `top_p` (nucleus sampling) keeps the smallest set of words whose combined probability reaches *p*.

Let's compare low vs. high temperature on the same prompt:

In [ ]:
test_prompt = "Give a summary of Alice in Wonderland"
gen_kwargs = dict(max_new_tokens=100, do_sample=True, return_full_text=False)

print("1. LOW TEMPERATURE (0.1):")
out_low = generator(test_prompt, temperature=0.1, **gen_kwargs)
print(out_low[0]['generated_text'])

print("\n" + "=" * 30 + "\n")

print("2. HIGH TEMPERATURE (1.5):")
out_high = generator(test_prompt, temperature=1.5, **gen_kwargs)
print(out_high[0]['generated_text'])

**✏️ Exercise: Parameter Playground**

Change `my_prompt` or `my_temperature` below and see how the model behaves.

In [ ]:
# --- PARAMETER PLAYGROUND ---
my_prompt = "Describe the smell of an old library."  # Try changing this!
my_temperature = 0.8  # Try values between 0.1 and 2.0
# -----------------------------

print(f"Generating with temperature: {my_temperature}...")
output_play = generator(my_prompt, max_new_tokens=150, do_sample=True,
                         temperature=my_temperature, return_full_text=False)
print("\n--- RESULT ---")
print(output_play[0]['generated_text'])

## Cheat sheet: more prompting tips

For a deeper dive, see ["Principled Instructions Are All You Need for Questioning LLaMA-1/2, GPT-3.5/4"](https://arxiv.org/abs/2312.16171) (Bsharat, Myrzakhan & Shen, 2023), summarised here:

| Category | Principle |
| --- | --- |
| Structure & clarity | Name the intended audience in the prompt. |
| Structure & clarity | Prefer affirmative directives ("do") over negative ones ("don't"). |
| Structure & clarity | Use leading phrases like "think step by step". |
| Structure & clarity | End the prompt with the start of the desired output (an "output primer"). |
| Structure & clarity | Use delimiters (`###`, triple quotes, XML tags…) to mark off input text. |
| Specificity | Give a few worked examples (few-shot prompting). |
| Specificity | Ask for clarification-level answers ("explain like I'm 11"). |
| Specificity | State requirements explicitly using keywords, constraints, or hints. |
| Content & style | Be direct and concise — skip unnecessary politeness. |
| Complex tasks | Break a complex task into a sequence of simpler prompts. |
| Complex tasks | Use structured prompts for long or multi-part tasks. |


# Part B — LLMs as Research Assistants

So far we've prompted the model one message at a time. Now we scale up: we'll turn the model into a small **research assistant** that reads, cleans, classifies, and structures hundreds of historical newspaper snippets.

**Language modelling** (see the previous notebook) is about how models *absorb* knowledge through pre-training. **Instruction following** — what we've been doing above — is what lets today's LLMs be steered with a prompt, using:
- **Few-shot / in-context learning**: "here are a few examples of what I want."
- **Retrieval-Augmented Generation (RAG)**: "here are some documents — base your answer on them."

We'll build a minimal version of the second approach — sometimes jokingly called **"baby RAG"**, or "elevated copy-pasting": we retrieve documents ourselves (with a simple keyword search) and paste them straight into the prompt, then ask the model to answer based only on that context.

## Why historical newspapers?

Newspapers are:
- **Big**: some of the largest digitised text collections available.
- **Fine-grained**: daily reports on everything from the banal to the momentous.
- **Longitudinal**: they run for decades, letting us trace change over time.

### Case study: accidents in the news

![](https://global.oup.com/academic/covers/pop-up/9780198732334)

Inspired by Paul Fyfe's *["By Accident or Design"](https://global.oup.com/academic/product/by-accident-or-design-9780198732334)*: newspapers are "periodical purveyors of miscellaneous content" — accident reports in particular are "subjective attempts at the construction of evidence" (Roger Cooter), and "useful as indices to social and especially industrial change" (Roger Lane). They also mention many "small" historical actors who otherwise leave little trace in the archive.

## Downloading a sample of newspaper articles

In [ ]:
import pandas as pd

# a small sample of digitised British newspaper articles, courtesy of the
# "Living with Machines" project: https://livingwithmachines.ac.uk/
df = pd.read_csv(
    "https://raw.githubusercontent.com/kasparvonbeelen/uga-llm-workshop/refs/heads/main/newspapers/0002247.csv",
    index_col=0,
)
print(df.shape)
df.head(3)

### Issues with the data

- **Segmentation**: it's not always obvious what counts as a single "article" — page layout and OCR segmentation are imperfect.
- **OCR quality**: historical print + automatic text recognition = lots of noise, as you can see below.

In [ ]:
print(df.iloc[0].content)

In [ ]:
print(df.iloc[10].content)

## Chunking the text

To make the material easier to work with (and to fit within the model's context window), we split each article into overlapping chunks of ~250 words. We first split on blank lines (a rough proxy for paragraph/article boundaries), then chunk each resulting piece.

In [ ]:
def get_chunks(text: str, size: int = 250, step: int = 50) -> list:
    """Divide a text into overlapping chunks of (roughly) equal size.
    Arguments:
      text (str): input text
      size (int): number of words per chunk
      step (int): step size between chunk start points (creates the overlap)
    Returns a list of strings.
    """
    words = text.split()
    return [' '.join(words[i:i + size]) for i in range(0, len(words), step)]

In [ ]:
# split each article on blank lines (a proxy for paragraph/article breaks)...
df['elements'] = df.content.apply(lambda x: [' '.join(ch.split('\n')) for ch in x.split('\n\n')])
df_by_element = df.explode('elements')

# ...then chunk each element, and put one chunk per row
df_by_element['chunks'] = df_by_element.elements.apply(get_chunks)
df_chunks = df_by_element.explode('chunks')
df_chunks.reset_index(drop=True, inplace=True)
df_chunks.shape  # that's a lot of chunks!

## Retrieving articles about accidents

The simplest possible retrieval step: a regular expression search for the word "accident".

In [ ]:
import re
from tqdm.auto import tqdm

pattern = re.compile(r'\baccidents?\b', re.I)
tqdm.pandas()
df_chunks['chunk_count'] = df_chunks.progress_apply(lambda x: len(pattern.findall(str(x['chunks']))), axis=1)
df_chunks.sort_values('chunk_count', ascending=False)[['title', 'chunk_count', 'chunks']][:5]

## Applying prompts to many documents at once

To process a whole DataFrame of chunks, we write one small helper, `apply_completions()`. Calling `get_completion()` once per row (e.g. via `df.apply(...)`) would work, but wastes the model's ability to process several inputs together — each call pays the full model overhead for just one row. Instead, `apply_completions()` builds one chat per row and sends the **whole batch** to the model in a single call, `batch_size` chats at a time: much faster, especially on a GPU.

In [ ]:
def apply_completions(df: pd.DataFrame, system_message: str, user_message: str = '',
                       text_column: str = 'chunks', batch_size: int = 8, **kwargs) -> list:
    """Apply one system/user prompt to every row of a DataFrame, processed in batches.
    Arguments:
      df (pd.DataFrame): rows to process
      system_message (str): system prompt — how the model should behave
      user_message (str): user prompt/instruction; each row's text is appended to it
      text_column (str): name of the column holding the text to process
      batch_size (int): how many rows the model processes together in one go
      **kwargs: passed on to the pipeline, e.g. max_new_tokens, temperature
    Returns a list of completions, one per row, in the same order as df.
    """
    chats = [
        [
            {"role": "system", "content": system_message},
            {"role": "user", "content": f"{user_message}\n\n###{text}###"},
        ]
        for text in df[text_column]
    ]
    outputs = generator(chats, batch_size=batch_size, do_sample=True,
                         max_new_tokens=kwargs.pop('max_new_tokens', 256),
                         temperature=kwargs.pop('temperature', 0.1),
                         top_p=kwargs.pop('top_p', 0.9), **kwargs)
    return [output[0]["generated_text"][-1]["content"] for output in outputs]

### Step 1: Classify (and clean)

Not every chunk that merely *mentions* "accident" is actually *about* an accident. We can use the model to **classify** (and, in a second pass, clean up the noisy OCR text of) each chunk, using few-shot examples.

Running this over many rows takes a little while even with a small model, so — to keep the workshop moving — we load a **precomputed** result below. The code that produced it is shown too, so you can run it yourself (on a subset, or with more time).

In [ ]:
# the 100 chunks that mention "accident" most often
df_accident = df_chunks.sort_values('chunk_count', ascending=False)[:100].reset_index(drop=True)

# precomputed classifications/cleaning/summaries/structured data for these
# same 100 chunks, so we don't all have to wait for generation during the workshop
df_accident = pd.read_json(
    'https://raw.githubusercontent.com/kasparvonbeelen/uga-llm-workshop/refs/heads/main/newspapers/df_accident.json'
)
df_accident[['title', 'chunk_count', 'chunks']].head()

In [ ]:
system_message = """You are a document classifier that will clean and correct
    snippets of newspaper articles as either about accidents or not about accidents.
    You answer with only 'yes', 'no', or 'unsure'.

    Examples are:
    Input: RAILWAY ACCIDENTS IN AMERICA. The record of fatal accidents on the railroads in Eng- land, which is published annually,
    Output: yes

    Input: LATEST INTELL IGEN CE. the prisoner was committed for trial for embezzlement. He was also further committed in two
    Output: no
    """

# uncomment to (re-)generate this yourself — slower, so it's commented out by default:
# df_accident['classification'] = apply_completions(df_accident, system_message)

df_accident['classification'].value_counts()

In [ ]:
system_message = """You are a document assistant that will clean and correct snippets of
    newspaper articles that mention the word 'accident'.
    Remove all text that is not about accidents, i.e. mentions of other events or facts.
    Try to correct OCR errors in the text wherever possible.

    Examples are:
    Input: "RAILWAY ACCIDENTS IN AMERICA. The record of fatal accidents on the railroads in Eng- land, which is published annually,
            POOR T,i,ENIPAT A 1„k CT  The Poor Law Coirdnissioti(rs have issued a ei; cular, dated the 20th instant, stating that they have consulted the Attorney and"
    Output: "RAILWAY ACCIDENTS IN AMERICA. The record of fatal accidents on the railroads in England, which is published annually."

    Input: "LATEST INTELL IGEN CE. the prisoner was committed for trial for embezzlement. He was also further committed in two"
    Output: ""
    """

# uncomment to (re-)generate this yourself:
# df_accident['clean'] = apply_completions(df_accident, system_message)

# strip the boilerplate the model sometimes adds (e.g. "Here is the cleaned text:\n\n")
df_accident['clean_article'] = df_accident['clean'].apply(lambda x: str(x).split(':\n\n')[-1].strip('#'))
df_accident['clean_article'].head()

**✏️ Exercise**

Adapt `system_message` above (or write a new one from scratch) to classify chunks by a different criterion relevant to your own research — e.g. crime vs. non-crime, or whether a named place is mentioned. Run it on a handful of rows with `apply_completions(df_accident.head(5), system_message)`.

In [ ]:
# Enter code here

### Step 2: Structuring information

Cleaning and classifying helps us **prepare** the data, but not yet **analyse** it. So far, RAG has been document-focused and dominated by a Q&A pattern — useful, but limited for questions like *"how did accidents change over time?"* or *"who is blamed for them?"*, which need data we can aggregate, not just more prose.

The fix: ask the model for **structured completions** — go from unstructured text straight to structured data. History is, among other things, about **people** — so let's extract short biographical entries (who was involved, and what happened to them) from each accident report.

In [ ]:
system_message = """
    You are an helpful AI for analysing historical newspaper articles.
    Extract biographical information from a newspaper article between ###.
    Return the biographical information as a list of Python dictionaries.
    The information MUST be extracted from the article, with spelling and wording identical to the source.
    This list of dictionaries should begin with a "START" tag and end with an "END" tag.
    Don't make things up! If you don't know the answer, return an empty list, i.e. [].

    Input:
    ###J. D. McPhill, a miner aged 58, died tragically in a railway accident, in which his wife M. M. McBilly, age 59, was also injured.###

    Output:
    START
    [
        {"name": "J. D. McPhill", "gender": "male", "profession": "miner", "age": 58, "outcome": "died", "cause": "railway accident"},
        {"name": "M. M. McBilly", "gender": "female", "age": 59, "outcome": "injured", "cause": "railway accident"}
    ]
    END

    Input:
    """

# a single example, live:
snippet = df_chunks.sort_values('chunk_count', ascending=False).iloc[1].chunks
messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": f"###{snippet}###"},
]
print(get_completion(messages, max_new_tokens=300))

Let's apply the same prompt across the whole (precomputed) sample:

In [ ]:
# uncomment to (re-)generate this yourself:
# df_accident['bio'] = apply_completions(df_accident, system_message, text_column='clean_article', max_new_tokens=300)

df_accident['bio'].head()

The model's reply is just a string. To turn it into a real Python object we can analyse, we write a small parsing function. We use `ast.literal_eval` (rather than the more dangerous built-in `eval`) since it only ever parses Python literals — safer when working with text generated by a model.

In [ ]:
import ast
from typing import List, Dict

def eval_completion(completion: str) -> List[Dict]:
    """Convert a model completion (a string) into a Python list of dictionaries.
    Returns an empty list if the completion can't be parsed."""
    try:
        return ast.literal_eval(completion.split('START')[-1].strip().rstrip('END').strip())
    except Exception:
        return []

df_accident['bio_structured'] = df_accident['bio'].apply(eval_completion)
df_accident['bio_structured'].head()

Now we can flatten these per-article lists into one tidy table — an excellent starting point for further (quantitative) analysis:

In [ ]:
df_bios = pd.DataFrame([d for _, row in df_accident.iterrows() for d in row['bio_structured']])
df_bios.head()

In [ ]:
df_bios.age.unique()

**✏️ Exercise**

Design your own structured-extraction schema for a different research question — e.g. extract `{"location": ..., "vehicle": ..., "cause": ...}` from each accident report, or adapt the whole prompt to a different genre of source (obituaries, court reports, shipping notices). Try it on `snippet` above, then (optionally) apply it to the full `df_accident` sample.

*(Note: this workshop deliberately uses small, fast models so everyone can run them on a laptop CPU. For real research, structured extraction usually needs a larger, more capable model — see "What if things don't work?" below.)*

In [ ]:
# Enter code here

# (Optional/Advanced) GraphRAG: from text to a knowledge graph

A recent idea worth knowing about is [GraphRAG](https://arxiv.org/abs/2404.16130): instead of (or in addition to) extracting a flat table, we convert a corpus into a **network of information**. A sentence like *"the driver injured the passenger"* becomes a (subject, predicate, object) triple: `(driver, injured, passenger)`. Clustering the resulting network can support answering complex questions ("what ages of people are involved in railway accidents?") across a whole collection at once. Here we focus on just the first step: converting text into triples.

In [ ]:
# reuse the same 100-chunk sample, precomputed clean/graph fields included
df_small = pd.read_json(
    'https://raw.githubusercontent.com/kasparvonbeelen/uga-llm-workshop/refs/heads/main/newspapers/df_graph.json'
)
df_small[['title', 'clean_article']].head()

In [ ]:
# prompt adapted from https://towardsdatascience.com/how-to-convert-any-text-into-a-graph-of-concepts-110844f22a1a
system_message = """You are a network graph maker who extracts terms and their relations from a given context.
    You are provided with a context chunk (delimited by ```). Your task is to extract the ontology of terms
    mentioned in the given context. These terms should represent the key concepts as per the context.

    Terms may include object, entity, location, organization, person, condition, acronym, document, service,
    concept, etc. Terms should be as atomistic as possible. Terms mentioned in the same sentence or paragraph
    are typically related; find the relation between each related pair of terms.

    Format your output as a list of JSON objects, each with a pair of terms and the relation between them:
    ```[
       {"node_1": "A concept from the extracted ontology",
        "node_2": "A related concept from the extracted ontology",
        "edge": "relationship between node_1 and node_2"},
       {...}
    ]```
    """

# uncomment to (re-)generate this yourself:
# df_small['graph'] = apply_completions(df_small, system_message, text_column='clean_article')

df_small['graph'].head(3)

In [ ]:
import ast

def eval_completion_graph(completion: str):
    try:
        return ast.literal_eval(completion.split('```')[1].strip())
    except Exception:
        return []

df_small['triples'] = df_small['graph'].apply(eval_completion_graph)

knowledge_graph = [
    (e['node_1'], e['edge'], e['node_2'])
    for triples in df_small['triples']
    for e in triples
]
graph_df = pd.DataFrame(knowledge_graph, columns=['node1', 'relation', 'node2'])
graph_df.head()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()
for node1, relation, node2 in knowledge_graph:
    G.add_edge(node1, node2, label=relation)

plt.figure(figsize=(14, 14), dpi=150)
pos = nx.kamada_kawai_layout(G, scale=3)
nx.draw_networkx_nodes(G, pos, node_size=400)
nx.draw_networkx_edges(G, pos, edge_color='gray', width=1.5)
nx.draw_networkx_labels(G, pos, font_size=10)
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, 'label'), font_size=8)
plt.axis('off')
plt.show()

**Question:** how would you improve this prompt? Researchers are still investigating how much forcing an LLM into a rigid output format (like our triples, or the biographical records above) reduces the quality of what it extracts — worth keeping in mind as you design your own structured prompts.

# What if things don't work?

Small, CPU-friendly models like the ones we used today are great for learning prompt engineering, but they will sometimes struggle with harder tasks (reliable structured output, subtle reasoning, long documents). If your own project needs more, a few options:

- **Use a larger open model** on a GPU, with the exact same `get_completion()`-style workflow used here — e.g. bump `model_id` up to `Qwen/Qwen2.5-7B-Instruct` (still ungated, no extra setup), or a different family entirely like `meta-llama/Meta-Llama-3-8B-Instruct` (gated; request access on its [model page](https://huggingface.co/meta-llama/Meta-Llama-3-8B) first).
- **Fine-tune** a model on your own (real or synthetic) data — see [this tutorial](https://huggingface.co/blog/mlabonne/sft-llama3) for a starting point.
- **Fall back to a closed, hosted model** (OpenAI, Anthropic, etc.) via their API — more capable out of the box, at a cost, and with your data leaving your machine. The cell below shows the shape of such a call (it needs a paid API key to run):

In [ ]:
# !pip install -q openai
#
# from openai import OpenAI
# client = OpenAI(api_key="sk-...")
#
# completion = client.chat.completions.create(
#     model="gpt-4o",
#     messages=[
#         {"role": "system", "content": "You are a helpful assistant. Correct the text below."},
#         {"role": "user", "content": df_chunks.iloc[4]["chunks"]},
#     ],
# )
# print(completion.choices[0].message.content)

# Fin.

Thanks for working through this notebook — now go prompt something historical! 🎉